# Extras: Reading and Writing CSV Files in Python

Next up we'll cover one of the most fundamental skills in data analysis: working with CSV (Comma-Separated Values) files. CSV is a universal format for storing tabular data, and Python is exceptionally good at handling it.

We will learn:
1.  How file paths work.
2.  How to write and read small CSV files.
3.  How to generate a massive dataset that is too big for Excel.
4.  How to use Python to analyze this large dataset effortlessly.

*Note: the Jupyter Notebook `Session-03-advanced-lists.ipynb` was our first introduction to the topics mentioned here.*

## 1. Where Are My Files? Understanding File Paths

First, where are my files going to go?

When you read or write a file in Python, the program needs to know where to look. If you just provide a filename like `my_data.csv`, Python will look in the **current working directory**.

The current working directory is the folder where your Jupyter Notebook or Python script is currently running. We can find it using the `os` module. The `os` module or library provides a portable way of using Operating System dependent functionality. So, high-level file and directory handling can be handled in a unified manner by your Python `.py` script without you having to know all the subtle quirks of the operating system your script is running on. See the [Python Documentation project for more detail](https://docs.python.org/3/library/os.html).

In the example below we call getcwd() method from the `os` module. `os.getcwd()` works as intended, it "gets the current working directory". 

n.b. `pwd` is the standard Unix terminal command for essentially the same thing (aka present working directory, aka print working directory). `getcwd` is the *Pythonic* way of invoking the appropriate OS equivalent command from within a Python script.

In [ ]:
import os

current_directory = os.getcwd()# call the function to get the current working directory
print(f"This notebook is running from: {current_directory}")
print("\nWhen we write a file, it will be saved in this folder unless we specify a different path.")

## 2. Creating a Simple CSV File

Let's start by creating a small CSV file using [Python's `csv` module](https://docs.python.org/3/library/csv.html). We'll create a simple list of lists, where each inner list represents a row of data for a stock trade.

First, we the import csv library, it provides all the necessary support for csv file creation, writing, reading, updating etc.


In [ ]:
import csv

# Our data: Header row and two data rows
trade_data = [
    ['Ticker', 'Quantity', 'Price'],
    ['AAPL', 100, 175.50],
    ['MSFT', 50, 305.25]
]

file_name = 'simple_trades.csv'

# 'with open(...)' is the standard way to handle files in Python.
# 'w' means we are opening the file in 'write' mode.
# newline='' prevents blank rows from being added.
with open(file_name, 'w', newline='') as csvfile:
    # Create a csv writer object
    writer = csv.writer(csvfile)
    
    # Write all the rows at once
    writer.writerows(trade_data)

print(f"Successfully created '{file_name}' in the current directory.")

## 3. Reading a Simple CSV File

Now, let's read the data back from the file we just created. We'll open it in 'read' mode (`'r'`) and loop through the rows. We will use a simple `for` loop to process the file.

In [ ]:
with open('simple_trades.csv', 'r') as csvfile:
    reader = csv.reader(csvfile)
    
    print("Reading data from simple_trades.csv:")
    for row in reader:
        print(row)

## 4. The Excel Killer: Generating a Large Dataset

That's great, a simple spreadsheet with three rows and three columns is trivial. 
Let's make a csv file with a non-trivial number of rows to demonstrate, that while spreadsheet programs like Excel are great, they do have limits. 
Most versions of Excel can't handle more than **1,048,576 rows**. 
What happens when your dataset is bigger?
The program will fail to open the file.

Let's demonstrate this by generating a CSV file with **1.5 million** stock trades. We'll use the `Faker` library to help create realistic-looking fake data, in combination with [Python's random module](https://docs.python.org/3/library/random.html) which implements pseudo-random number generation for various distributions. 

First, you may need to install faker: `pip install Faker`

In the first instance we'll code for a random price across all stocks between $100 and $500, rounded to 2 decimals.

```python
price = round(random.uniform(100, 500), 2)
```
Later, we can experiment with different distributions such as uniform, normal (Gaussian), lognormal, negative exponential, gamma, and beta distributions.

In [ ]:
from faker import Faker
import random # Python's random module, used for generating random numbers

fake = Faker()
large_file_name = 'large_trade_log.csv'
num_rows = 1_500_000 # 1.5 million rows
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA', 'JPM', 'V']

print(f"Generating {num_rows:,} rows of trade data... this may take a moment.")

with open(large_file_name, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Write header
    writer.writerow(['Timestamp', 'Ticker', 'Price', 'Volume'])
    
    for _ in range(num_rows):
        timestamp = fake.date_time_between(start_date='-1y', end_date='now').isoformat()
        ticker = random.choice(tickers)
        price = round(random.normalvariate(100, 500), 2) # Random price between $100 and $500, rounded to 2 decimals.
        volume = random.randint(10, 1000)
        writer.writerow([timestamp, ticker, price, volume])

file_size = os.path.getsize(large_file_name) / (1024*1024) # in MB
print(f"\nSuccessfully created '{large_file_name}' ({file_size:.2f} MB).")
print("Try opening this file in Excel! It will likely fail.")

## 5. Analyzing the Large Dataset with Python

We can inspect the file directly and easily with **Pandas** (we'll cover in detail in session 7). Pandas is the standard tool for manipulating data collections and uses NumPy arrays under the hood.

Let's read our massive CSV into a pandas DataFrame.

In [ ]:
import pandas as pd

print("Reading the 1.5 million row CSV file into a pandas DataFrame...")
# pandas is highly optimized for reading large CSVs
trades_df = pd.read_csv(large_file_name)

print("File read successfully!")
print(f"Shape of the DataFrame (rows, columns): {trades_df.shape}")

### 5a. Inspecting your Data with the `info`, `describe`, and `head` Methods

Now that the data is loaded into a pandas DataFrame, aside from the shape, what's inside? 
Before doing any calculations, it is standard practice to inspect the data. 
This helps you verify that the data loaded correctly, understand the data types of your columns, cou can check for missing values, and get a high-level statistical overview.

In [ ]:
# 1. Preview the first few rows with .head()
# This is the best way to get a quick look at the columns and the kind of data they contain.
print("--- First 5 Rows (Preview using .head()) ---")
print(trades_df.head())
print("\n")

# 2. Get a technical summary with .info()
# This is crucial for checking the number of non-null (i.e., not empty) entries for each column
# and what data type pandas assigned to each column (e.g., object for text, float64 for decimals).
print("--- Technical Summary (using .info()) ---")
trades_df.info()
print("\n")

# 3. Get a statistical summary with .describe()
# This provides key descriptive statistics for all numerical columns, like mean, standard deviation, min, max, and quartiles.
print("--- Statistical Summary (using .describe()) ---")
print(trades_df.describe())

### 5b. Commence Your Analysis

Now that we've inspected our data and confirmed it's what we expect, we can proceed with the analysis we planned. This is where NumPy excels! (pun intended). Observe how quickly the analysis executes. NumPy  calculates the `sum` of all trades, followed by code that creates a Pandas DataFrame containing just the NVDA trades, roughly 25% of the 1.5million row table and finally, NumPy is called to calculate the mean. 

It takes approximately 0.1s to do all that.

In [ ]:
# Calculate the total volume of all trades
# We can use NumPy's sum function on the underlying array for efficiency
total_volume_traded = np.sum(trades_df['Volume'].values)
print(f"Total shares traded across all stocks: {total_volume_traded:,}")

# Find the average trade price for a specific stock (e.g., NVDA)
nvda_trades = trades_df[trades_df['Ticker'] == 'NVDA']
average_nvda_price = np.mean(nvda_trades['Price'].values)
print(f"Average trade price for NVDA: ${average_nvda_price:.2f}")

## Hands-on Recap

We can create, write and read `csv` files with python. We know where our files are stored and accessed on your local operating system using `os`. We have also used a data generator package (`Faker`) and Python's `random` library to create large realistic synthetic datasets for testing. We have see instances where Python (using `NumPy`, `Pandas` and friends) can be superior to `Excel`, particularly cases with the large datasets typical in finance such as historic stock exchange trading data.

You have now had hands-on experience across the entire workflow of handling CSV files in Python. 

You can now handle datasets of virtually any size, perform complex calculations and conduct analysis that are simply out of reach for traditional spreadsheet software. This is a key reason why Python is a crucial tool for science, finance and economics.